In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./raw_data.csv")

In [3]:
df.head()

,Season,Competition,Matchday,Date,Venue,Club,Opponent,Result,Playing_Position,Minute,At_score,Type,Goal_assist
0,04/05,LaLiga,34,05-01/05,H,FC Barcelona,Albacete Balompie,2:00,CF,90+1,2:00,Left-footed shot,Ronaldinho Gaacho
1,05/06,UEFA Champions League,Group Stage,11-02/05,H,FC Barcelona,Panathinaikos Athens,5:00,RW,34,3:00,Left-footed shot,NaN
2,05/06,LaLiga,13,11/27/05,H,FC Barcelona,Racing Santander,4:01,RW,51,2:00,Left-footed shot,Samuel Etoo
3,05/06,LaLiga,19,1/15/06,H,FC Barcelona,Athletic Bilbao,2:01,RW,50,2:01,Left-footed shot,Mark van Bommel
4,05/06,LaLiga,20,1/22/06,H,FC Barcelona,Deportivo Alaves,2:00,CF,82,2:00,Left-footed shot,Ronaldinho Gaacho


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 704 entries, 0 to 703
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Season            704 non-null    object
 1   Competition       704 non-null    object
 2   Matchday          704 non-null    object
 3   Date              704 non-null    object
 4   Venue             704 non-null    object
 5   Club              704 non-null    object
 6   Opponent          704 non-null    object
 7   Result            704 non-null    object
 8   Playing_Position  704 non-null    object
 9   Minute            704 non-null    object
 10  At_score          704 non-null    object
 11  Type              703 non-null    object
 12  Goal_assist       490 non-null    object
dtypes: object(13)
memory usage: 71.6+ KB


In [5]:
df['Club'].value_counts()

Club
FC Barcelona           672
Paris Saint-Germain     32
Name: count, dtype: int64

In [6]:
df['Opponent'].value_counts()

Opponent
Sevilla FC             38
Atletico de Madrid     32
Valencia CF            31
Athletic Bilbao        29
Real Madrid            26
                       ..
ESTAC Troyes            1
Angers SCO              1
Montpellier HSC         1
FC Toulouse             1
Olympique Marseille     1
Name: count, Length: 98, dtype: int64

In [11]:
df['Minute'].dtype

dtype('O')

Transform the `Minute` column so that it meets my needs. Instead of numbers representing minutes, I will transform them into first, second half, additional time and extra time

In [18]:
import numpy as np

# helper function to clean 'Minute' column
def convert_minute(minute_str):
    if pd.isna(minute_str):
        return None
    if '+' in minute_str:
        base, _ = minute_str.split('+')
        base = int(base)
        if base <= 45:
            return "First half"
        elif base <= 90:
            return "Additional time"
        else:
            return "Extra time"
    
    minute = int(minute_str)
    if minute <= 45:
        return "First half"
    elif minute <= 90:
        return "Second half"
    else:
        return "Extra time"

df['Minute'] = df['Minute'].apply(convert_minute)

In [13]:
df['Date']

0      05-01/05
1      11-02/05
2      11/27/05
3       1/15/06
4       1/22/06
         ...   
699      2/1/23
700      2/4/23
701     2/19/23
702     2/26/23
703      3/4/23
Name: Date, Length: 704, dtype: object

We need to standardize this `Date` column

In [17]:
def standardize_date(date_str):
    from dateutil import parser
    try:
        date_str = date_str.replace('-','/')
        dt = parser.parse(date_str,dayfirst=False,yearfirst=False)
        return dt.strftime("%Y-%m-%d")
    except:
        return None

df['Date'] = df['Date'].apply(standardize_date)

In [15]:
df['Venue']

0      H
1      H
2      H
3      H
4      H
      ..
699    A
700    H
701    H
702    A
703    H
Name: Venue, Length: 704, dtype: object

In [16]:
df['Venue'] = df['Venue'].map({'H': 'Home', 'A': 'Away'}).fillna(df['Venue'])
df['Venue'].unique()

array(['Home', 'Away'], dtype=object)

In [19]:
# replace NaN with 'None' string across the DataFrame
df = df.fillna('None')

In [20]:
df['Competition'].unique()

array(['LaLiga', 'UEFA Champions League', 'Copa del Rey', 'Supercopa',
       'FIFA Club World Cup', 'UEFA Super Cup', 'Ligue 1',
       'Trophée des Champions', 'Troph�e des Champions',
       'Champions League'], dtype=object)

In [21]:
df['Competition'] = df['Competition'].replace({
    'Champions League': 'UEFA Champions League',
    'Troph�e des Champions': 'Trophée des Champions'
})

In [23]:
df['At_score'].head()

0    2:00
1    3:00
2    2:00
3    2:01
4    2:00
Name: At_score, dtype: object

In [26]:
# remove leading zeros from both sides
def clean_score(score_str):
    if score_str == 'None':
        return score_str
    try:
        parts = score_str.split(':')
        left = str(int(parts[0].strip()))
        right = str(int(parts[1].strip()))
        return f"{left} : {right}"
    except:
        return 'None'
    
df['At_score'] = df['At_score'].apply(clean_score)
df['Result'] = df['Result'].apply(clean_score)

df['Season'] = df['Season'].astype('string')

In [27]:
df['Season'].unique()

<StringArray>
[ '04/05',  '05/06',  '06/07',  '07/08',  '08/09',  '09/10',  '10/11',
 '11-Dec', 'Dec-13',  '13/14',  '14/15',  '15/16',  '16/17',  '17/18',
  '18/19',  '19/20',  '20/21',  '21/22',  '22/23']
Length: 19, dtype: string

In [28]:
import re

def fix_season_format(season):
    if isinstance(season, str):
        # match cases like "Dec-13" or "11-Dec"
        if re.match(r'^(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)[- ]\d{2}$', season):
            # Convert "Dec-13" to "12/13"
            month_str, year_str = season.split('-')
            start_year = {
                'Aug': '08', 'Sep': '09', 'Oct': '10', 'Nov': '11', 'Dec': '12',
                'Jan': '01', 'Feb': '02', 'Mar': '03', 'Apr': '04', 'May': '05', 'Jun': '06', 'Jul': '07'
            }.get(month_str[:3], '00')
            end_year = year_str
            return f"{start_year}/{end_year}"
        
        elif re.match(r'^\d{2}[- ](Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$', season):
            # Convert "11-Dec" to "11/12"
            start_year = season[:2]
            return f"{start_year}/{str(int(start_year)+1).zfill(2)}"
        
        elif re.match(r'^\d{4}-\d{2}-\d{2}$', season):
            # Skip full dates like "2011-07-01"
            return "None"
    return season

# Apply the function
df['Season'] = df['Season'].apply(fix_season_format).astype('string')


In [30]:
df['Playing_Position'].unique()

array(['CF', 'RW', 'LW', 'SS', 'CF ', 'AM', 'RW ', 'AM ', 'SS '],
      dtype=object)

In [31]:
df['Type'].unique()

array(['Left-footed shot', 'Right-footed shot', 'Header', 'Solo run',
       'None', 'Penalty', 'Deflected shot on goal', 'Direct free kick',
       'Penalty rebound', 'Counter attack goal', 'Chest', 'Tap-in',
       'Long distance kick'], dtype=object)

In [32]:
df['Matchday'].unique()

array(['34', 'Group Stage', '13', '19', '20', '21', 'Quarter-Finals',
       '24', '1', '2', '6', '26', '27', '28', 'Semi-Finals', '35', '37',
       '38', '4', '5', '7', '9', 'last 16', '25', '3', 'Fifth Round',
       '11', '15', 'Round of 16', '18', '31', '33', 'Final', 'final', '8',
       '10', '5th round', '17', '30', '36', '4th round', '12', '14', '22',
       '32', '29', '16', '23'], dtype=object)

In [36]:
def clean_matchday(x):
    if pd.isna(x):
        return x

    x = str(x).strip()

    # purely numeric
    if re.fullmatch(r"\d+", x):
        return "Regular Season"

    if x.lower() == "final":
        return "Final"

    if x == "Fifth Round":
        return "5th round"
    
    if x == 'last 16':
        return 'Round of 16'

    return x

df['Matchday'] = df['Matchday'].apply(clean_matchday)

In [37]:
df['Matchday'].unique()

array(['Regular Season', 'Group Stage', 'Quarter-Finals', 'Semi-Finals',
       'Round of 16', '5th round', 'Final', '4th round'], dtype=object)

In [38]:
df.head()

,Season,Competition,Matchday,Date,Venue,Club,Opponent,Result,Playing_Position,Minute,At_score,Type,Goal_assist
0,04/05,LaLiga,Regular Season,2005-05-01,Home,FC Barcelona,Albacete Balompie,2 : 0,CF,Additional time,2 : 0,Left-footed shot,Ronaldinho Gaacho
1,05/06,UEFA Champions League,Group Stage,2005-11-02,Home,FC Barcelona,Panathinaikos Athens,5 : 0,RW,First half,3 : 0,Left-footed shot,None
2,05/06,LaLiga,Regular Season,2005-11-27,Home,FC Barcelona,Racing Santander,4 : 1,RW,Second half,2 : 0,Left-footed shot,Samuel Etoo
3,05/06,LaLiga,Regular Season,2006-01-15,Home,FC Barcelona,Athletic Bilbao,2 : 1,RW,Second half,2 : 1,Left-footed shot,Mark van Bommel
4,05/06,LaLiga,Regular Season,2006-01-22,Home,FC Barcelona,Deportivo Alaves,2 : 0,CF,Second half,2 : 0,Left-footed shot,Ronaldinho Gaacho


In [43]:
# save the cleaned data
df.to_excel("./cleaned_data.xlsx",index=False)

In [42]:
!pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

